# 03 · Silver → Gold: Modelagem Dimensional (Dimensões de Atendimento)

Neste notebook construímos as tabelas da camada **Gold** sob a ótica da **Modelagem Dimensional (*Star Schema*)** para o Case V-Credit.

### Objetivo desta etapa:
- Consumir as tabelas limpas e padronizadas da camada Silver (`vcredit.silver.tb_atendentes` e `vcredit.silver.tb_motivos`).
- Aplicar regras analíticas de negócio, como o cálculo de **`tempo_empresa` (em dias)**.
- Disponibilizar as dimensões com nomenclatura de consumo analítico (`dim_atendente` e `dim_motivo`) prontas para relacionamento direto com a tabela fato no **Power BI**.
- Garantir idempotência e integridade na gravação em formato **Delta Lake**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

catalog = "vcredit"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Lendo da Silver: {silver_schema}")
print(f"Gravando na Gold: {gold_schema}")

---
## 1. Dimensão Atendente (`vcredit.gold.dim_atendente`)

### Regra de Negócio e Enriquecimento:
A dimensão de atendente requer os atributos cadastrais e a métrica de tempo de permanência na empresa:
- **`id_atendente`**: Chave primária da dimensão (`PK`);
- **`nome_atendente`**, **`nivel_atendimento`**, **`status_atendente`**;
- **`data_admissao`** e **`data_desligamento`**;
- **`tempo_empresa` (em dias)**: calculado dinamicamente:
  - Se o atendente foi desligado (`data_desligamento IS NOT NULL`): `datediff(data_desligamento, data_admissao)`.
  - Se o atendente continua ativo (`data_desligamento IS NULL`): `datediff(current_date(), data_admissao)`.

In [0]:
df_silver_atendentes = spark.table(f"{silver_schema}.tb_atendentes")

df_gold_atendente = (
    df_silver_atendentes
    .withColumn(
        "tempo_empresa",
        F.when(
            F.col("data_desligamento").isNotNull(),
            F.datediff(F.col("data_desligamento"), F.col("data_admissao"))
        ).otherwise(
            F.datediff(F.current_date(), F.col("data_admissao"))
        )
    )
    .select(
        F.col("id_atendente"),
        F.col("nome_atendente"),
        F.col("nivel_atendimento"),
        F.col("data_admissao"),
        F.col("data_desligamento"),
        F.col("status_atendente"),
        F.col("tempo_empresa").cast(IntegerType()).alias("tempo_empresa")
    )
)

(
    df_gold_atendente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_atendente")
)

print(f"Tabela {gold_schema}.dim_atendente salva com sucesso!")
print(f"Total de atendentes na dimensão: {df_gold_atendente.count()}")
display(df_gold_atendente)
